1. Test the URL

In [1]:
import requests
import pandas as pd 
import io 

url = "https://www.fao.org/media/docs/worldfoodsituationlibraries/default-document-library/food_price_indices_data.csv?sfvrsn=523ebd2a_80&download=true"

response = requests.get(url, timeout=30)
print("Status code:", response.status_code)
print("First 500 characters of content:")
print(response.text[:500])

Status code: 200
First 500 characters of content:
FAO Food Price Index,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2014-2016=100,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Date,Food Price Index,Meat,Dairy,Cereals,Oils,Sugar,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1990-01,64.4,74.3,53.5,64.1,44.59,87.9,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1990-02,64.7,76.8,52.2,62.2,44.50,90.7,,,,,,,,,,,,,,,,,


2. Load and clean the csv

In [2]:
df_raw = pd.read_csv(
    io.StringIO(response.text),
    skiprows=3,
    usecols=[0,1,2,3,4,5],
    names=["date_str", "food_price_index", "meat", "dairy", "cereals", "oils"],
    header=0,
)

print(f"Shape: {df_raw.shape}")
print("\nFirst 5 rows:")
print(df_raw.head())
print("\nLast 5 rows:")
print(df_raw.tail())
print(f"\nNull values: \n{df_raw.isnull().sum()}")

Shape: (437, 6)

First 5 rows:
  date_str  food_price_index  meat  dairy  cereals   oils
0  1990-01              64.4  74.3   53.5     64.1  44.59
1  1990-02              64.7  76.8   52.2     62.2  44.50
2  1990-03              64.0  78.5   41.4     61.3  45.75
3  1990-04              66.0  81.2   48.4     62.8  44.02
4  1990-05              64.6  81.8   39.2     62.0  45.50

Last 5 rows:
    date_str  food_price_index   meat  dairy  cereals   oils
432  2026-01             124.1  124.9  120.7    107.5  168.6
433  2026-02             125.5  126.5  119.4    108.7  174.2
434  2026-03             128.7  128.0  121.2    110.4  183.1
435  2026-04             131.0  130.4  119.7    111.3  193.9
436  2026-05             130.8  130.5  119.2    114.3  185.0

Null values: 
date_str            0
food_price_index    0
meat                0
dairy               0
cereals             0
oils                0
dtype: int64


3. Parse dates and filter to 2015 onwards

In [ ]:
# Convert "1990-01" format to proper datetime
df_raw["month_date"] = pd.to_datetime(df_raw["date_str"], format="%Y-%m")

# Standardise to first of month
df_raw["month_date"] = df_raw["month_date"].dt.to_period("M").dt.to_timestamp()

# Filter to 2015 onwards — aligns with all other sources
df = df_raw[df_raw["month_date"] >= "2015-01-01"].copy()

# Keep only date and oils
df = df[["month_date", "oils"]].rename(columns={"oils": "fao_veg_oil_index"})

print(f"Shape: {df.shape}")
print(f"Date range: {df["month_date"].min()} to {df["month_date"].max()}")
print()
print(df.tail(6).to_string())

Shape: (137, 2)
Date range: 2015-01-01 00:00:00 to 2026-05-01 00:00:00

    month_date  fao_veg_oil_index
431 2025-12-01              165.2
432 2026-01-01              168.6
433 2026-02-01              174.2
434 2026-03-01              183.1
435 2026-04-01              193.9
436 2026-05-01              185.0


4. Load World Bank data and check correlation

In [4]:
df_wb = pd.read_csv("nb01_wb_prices_clean.csv", parse_dates=["month_date"])
df_wb["month_date"] = df_wb["month_date"].dt.to_period("M").dt.to_timestamp()

# Join FAO oils index with World Bank CPO price
df_corr = df.merge(df_wb[["month_date", "cpo_price"]], on="month_date", how="inner")

print("Shape after join:", df_corr.shape)
print()

# Correlation check
corr = df_corr["fao_veg_oil_index"].corr(df_corr["cpo_price"])
print(f"Correlation between FAO Vegetable Oil Index and CPO price: {corr:.4f}")
print()

# Also check against all four oils
for col in ["cpo_price", "soyoil_price", "sunflower_price", "rapeseed_price"]:
    df_full = df.merge(df_wb[["month_date", col]], on="month_date", how="inner")
    c = df_full["fao_veg_oil_index"].corr(df_full[col])
    print(f"  FAO Oils Index vs {col}: {c:.4f}")

Shape after join: (137, 3)

Correlation between FAO Vegetable Oil Index and CPO price: 0.9648

  FAO Oils Index vs cpo_price: 0.9648
  FAO Oils Index vs soyoil_price: 0.9323
  FAO Oils Index vs sunflower_price: 0.9526
  FAO Oils Index vs rapeseed_price: 0.9335


5. Save output csv

In [5]:
output_cols = [
    "month_date",
    "fao_veg_oil_index"
]
 
df_corr[output_cols].to_csv("nb05_fao_vegoil_clean.csv", index=False)
print("Saved: nb05_fao_vegoil_clean.csv")
print(f"Shape: {df_corr[output_cols].shape}")
print(f"Columns: {output_cols}")

Saved: nb05_fao_vegoil_clean.csv
Shape: (137, 2)
Columns: ['month_date', 'fao_veg_oil_index']


## Decision: KEEP

Correlation with CPO price: 0.9648
Correlation with all substitutes: 0.93–0.96

Despite exceeding the 0.90 drop threshold, keeping for two reasons:
1. Single number that summarises the entire vegetable oil complex — useful as dashboard KPI
2. High correlation is a finding in itself: confirms CPO is the dominant driver of global
   veg oil pricing. Useful analytical narrative for SD Guthrie context.

FAO index will NOT be used in Panel 3 spread charts — World Bank prices are used directly.
FAO index used only as supplementary KPI card / context annotation.

Output: nb05_fao_vegoil_clean.csv — 137 rows, 2015-01-01 to 2026-05-01, no nulls.